# Programmable Photonic Transformations

Can a programmable 2×2 photonic circuit implement a target complex optical transformation, and how do component-level imperfections affect its accuracy and transmission?

## Optical Linear Transformations

Under the  `10_directional_couplers_and_coupled_mode_theory`, `11_mach_zehnder_interferometers`

### Optical fields as complex vectors

Each component represents a power-normalized complex modal amplitude, such that its squared magnitude corresponds to the optical power carried by the associated mode.
$$\mathbf{a}_\text{in} = \begin{bmatrix}
a_1 \\
a_2
\end{bmatrix} \quad \mathbf{a}_\text{out} = \begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}$$

$$P_\text{in} = \mathbf{a}_\text{in}^\dagger \mathbf{a}_\text{in}$$

$$\mathbf{a}_\text{out} = U\mathbf{a}_\text{in}$$

lossless
$$U^\dagger U = I$$

so,
$$P_\text{out} = \mathbf{a}_\text{in}^\dagger U^\dagger U \mathbf{a}_\text{in} = P_\text{in}$$

Therefore, 
$$\Vert \mathbf{a}_\text{out} \Vert^2 = \Vert \mathbf{a}_\text{in} \Vert^2$$

$$C(\theta) = \begin{bmatrix}
\cos\theta & -i\sin\theta \\
-i\sin\theta & \cos\theta
\end{bmatrix}$$

$$P(\phi) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\phi}
\end{bmatrix}$$

Ideally, $50:50$ coupler MZI is
$$U_\text{MZI}(\theta_1, \theta_2, \phi) = C(\theta_2)P(\phi)C(\theta_1) $$
$\theta_1 = \theta_2 = \frac{\pi}{4}$

An MZI that adjusts only a single internal relative phase can describe only a limited set of matrices. However, a $2 \times 2$ unitary matrix generally possesses four real degrees of freedom. If we disregard the global phase, there are three physically distinct degrees of freedom. Given this limitation, let us introduce additional phase shifters—beyond the internal phase—to control the relative phases at the input and output ports, thereby expanding the range of implementable transformations.

Let's define additional phase shifters:
$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

By adding input and output phase shifters to an ideal 50:50 MZI, one can represent any $2\times2$ unitary matrix, up to a global phase. In this note, Ignore the global phase of the independent 2-port transformation.
ignore
then, we can get
$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

when $U_t$ is unitary,
$$F_\text{shape} = \frac{|\mathrm{Tr}(U_t^\dagger U)|^2}{2\,\mathrm{Tr}(U^\dagger U)}$$

$$T_\text{avg} = \frac{\mathrm{Tr}(U^\dagger U)}{2}$$

and also,
$$U(\mathbf{p}) \approx e^{i\chi}U_t$$
$$\mathbf{p} = (\alpha, \phi, \beta)$$
This means that by varying controllable phases, a target matrix can be implemented—up to a global phase.

We need these indicators because optical losses can occur, even while the target complex matrix form is perfectly preserved.

$$U = \sqrt{\eta}\,e^{i\chi}U_t$$

$$F_\text{shape} = 1, \quad T_\text{avg} = \eta$$

### Model assumptions
- Linear, coherent, power-normalised two-mode system

- Ideal 50:50 directional couplers

- Independently controllable relative phase shifts

- No loss in the baseline unitary model

In [1]:
from src.verification import absolute_error, complex_error, relative_error, find_convergence_threshold
from src.waveguide import create_uniform_grid, slab_index_profile, second_derivative_matrix, slab_waveguide_operator, solve_eigenmodes, select_guided_modes
from src.coupler import double_slab_cell_average
from src.mzi import mzi_matrix, phase_matrix

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import math


COLORS = {
    "numerical": "#1F5A94",
    "numerical_alt": "#4C78A8",
    
    "analytical": "#D97706",
    "approximation": "#2E7D32",
    
    "error": "#7E57C2",
    "error_alt": "#B07AA1",

    "reference": "#6B7280",
    "reference_alt": "#9CA3AF"
}

figure_dir = Path("../figures/12_programmable_photonic_transformations")
figure_dir.mkdir(parents=True, exist_ok=True)


## Case 1 — Target Unitary Synthesis

Can a programmable $2 \times 2$ photonic circuit reproduce a prescribed complex unitary transformation using independently controllable phase shifts?

임의의 입력 복소 진폭 벡터에 대해 목표 행렬과 동일한 선형 변환을 구현할 수 있는가?

$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

control variant
$$\mathbf{p} = (\alpha, \phi, \beta)$$


$$U(\alpha,\phi,\beta) = D_{\mathrm{out}}(\alpha)U_{\mathrm{MZI}}(\phi)D_{\mathrm{in}}(\beta)$$

$$U(\alpha,\phi,\beta) = \begin{bmatrix}
1 & 0\\0 & e^{i\alpha}
\end{bmatrix}\frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})\\
-i(1+e^{i\phi}) & e^{i\phi}-1
\end{bmatrix}\begin{bmatrix}
1 & 0\\
0 & e^{i\beta}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta) = \frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})e^{i\beta}\\
-i(1+e^{i\phi})e^{i\alpha} & (e^{i\phi}-1)e^{i(\alpha+\beta)}
\end{bmatrix}
$$

Using Euler's formula,

$$1-e^{i\phi}=-2ie^{i\phi/2}\sin\frac{\phi}{2}$$

$$1+e^{i\phi}=2e^{i\phi/2}\cos\frac{\phi}{2}$$

Therefore,

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U^\dagger (\alpha,\phi,\beta) = e^{-i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{-i\alpha}\cos\frac{\phi}{2}\\
e^{-i\beta}\cos\frac{\phi}{2} & -e^{-i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
\sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2} & e^{-i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) \\
e^{-i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) & \sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}$$

$$U^\dagger U = I$$

**Goal**:

$$U_t = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{i\pi / 3} \\
e^{i\pi / 4} & -e^{i7\pi / 12}
\end{bmatrix}$$

$$
U_t^{\dagger} = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{-i\pi / 4} \\
e^{-i\pi / 3} & -e^{-i7\pi / 12}
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = \frac{1}{2}\begin{bmatrix}
1+1 & 0 \\
0 & 1+1
\end{bmatrix}$$

$$U_t^{\dagger}U_t = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = I$$

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta)= U_t$$

$$\alpha = \frac{\pi}{4}, \quad \beta = \frac{\pi}{3}, \quad \phi = \frac{\pi}{2}$$

In [10]:
def programmable_matrix(alpha, phi, beta):
    theta = np.pi / 4

    D_in = phase_matrix(beta)
    D_out = phase_matrix(alpha)
    U_mzi = mzi_matrix(theta, theta, phi)

    return D_out @ U_mzi @ D_in


In [11]:
alpha = np.pi / 4
phi = np.pi / 2
beta = np.pi / 3

U = programmable_matrix(alpha, phi, beta)

# Prescribed target
U_target = np.array([
    [1, np.exp(1j * np.pi / 3)],
    [np.exp(1j * np.pi / 4), -np.exp(1j * 7 * np.pi / 12)]
], dtype=complex) / np.sqrt(2)

print("Implemented U:")
print(U)

print("\nTarget U:")
print(U_target)


Implemented U:
[[ 0.5       -0.5j         0.6830127 +0.1830127j ]
 [ 0.70710678+0.j         -0.35355339-0.61237244j]]

Target U:
[[0.70710678+0.j         0.35355339+0.61237244j]
 [0.5       +0.5j        0.1830127 -0.6830127j ]]


In [7]:
I = np.eye(2, dtype=complex)

target_residual = np.linalg.norm(U_target.conj().T @ U_target - I, ord="fro")

unitarity_residual = np.linalg.norm(U.conj().T @ U - I, ord="fro")

overlap = np.trace(U_target.conj().T @ U)
power_trace = np.trace(U.conj().T @ U).real

F_shape = np.abs(overlap)**2 / (2 * power_trace)
T_avg = power_trace / 2

print(f"Target unitarity residual: {target_residual:.6e}")
print(f"Circuit unitarity residual: {unitarity_residual:.6e}")
print(f"Shape fidelity: {F_shape:.12f}")
print(f"Average transmission: {T_avg:.12f}")

Target unitarity residual: 1.766354e-16
Circuit unitarity residual: 6.280370e-16
Shape fidelity: 1.000000000000
Average transmission: 1.000000000000


In [9]:
chi_est = np.angle(overlap)

U_aligned = np.exp(-1j * chi_est) * U

aligned_error = (
    np.linalg.norm(U_aligned - U_target, ord="fro")
    / np.linalg.norm(U_target, ord="fro")
)

print(f"Estimated global phase: {chi_est:.12f} rad")
print(f"Phase-aligned matrix error: {aligned_error:.6e}")

assert unitarity_residual < 1e-12
assert abs(F_shape - 1.0) < 1e-12
assert abs(T_avg - 1.0) < 1e-12
assert aligned_error < 1e-12

Estimated global phase: -0.785398163397 rad
Phase-aligned matrix error: 3.223032e-16


$$U = e^{i\chi}\begin{bmatrix}
s & ce^{i\beta} \\
ce^{i\alpha} & -se^{i(\alpha + \beta)}
\end{bmatrix}$$

$$s = \sin\frac{\phi}{2}, \quad c = \cos\frac{\phi}{2}$$

$$\alpha = \arg\left(\frac{u_{21}}{u_{11}}\right), \quad \beta = \arg\left(\frac{u_{12}}{u_{11}}\right)$$

In [17]:
def synthesize_phases(U_target):

    alpha = np.angle(U_target[1, 0] / U_target[0, 0])
    beta = np.angle(U_target[0, 1] / U_target[0, 0])

    phi = 2 * np.arctan2(np.abs(U_target[0, 0]),np.abs(U_target[0, 1]))

    return alpha, phi, beta

In [18]:
alpha_fit, phi_fit, beta_fit = synthesize_phases(U_target)

print("alpha:", alpha_fit, "rad")
print("phi:", phi_fit, "rad")
print("beta:", beta_fit, "rad")

U_reconstructed = programmable_matrix(alpha_fit, phi_fit, beta_fit)

overlap = np.trace(U_target.conj().T @ U_reconstructed)
chi = np.angle(overlap)

error = np.linalg.norm(np.exp(-1j * chi) * U_reconstructed - U_target, ord="fro")

print("Reconstruction error:", error)


alpha: 0.7853981633974483 rad
phi: 1.5707963267948966 rad
beta: 1.0471975511965976 rad
Reconstruction error: 4.558056183912393e-16


In [21]:
def synthesize_phases_new(U_target, tol=1e-12):

    U_target = np.asarray(U_target, dtype=complex)

    if U_target.shape != (2, 2):
        raise ValueError("Target must be a 2x2 matrix.")

    if not np.allclose(U_target.conj().T @ U_target, np.eye(2), atol=tol, rtol=0):
        raise ValueError("Target must be unitary.")

    # Off-diagonal transformation
    if np.abs(U_target[0, 0]) < tol:
        
        alpha = np.angle(U_target[1, 0] / U_target[0, 1])
        beta = 0.0

        phi = 0.0

    # Diagonal transformation
    elif np.abs(U_target[0, 1]) < tol:
        
        alpha = np.angle(-U_target[1, 1] / U_target[0, 0])
        beta = 0.0

        phi = np.pi

    else:
        
        alpha = np.angle(U_target[1, 0] / U_target[0, 0])
        beta = np.angle(U_target[0, 1] / U_target[0, 0])

        phi = 2 * np.arctan2(np.abs(U_target[0, 0]), np.abs(U_target[0, 1]) )

    return alpha, phi, beta